# MouserCV - Notebook 3: Full Video Processing Pipeline

This notebook runs the **complete MouserCV processing pipeline** on a single video:

1. **SAM3** segmentation and tracking
2. **SuperAnimal** keypoint detection
3. **Feature extraction** from masks
4. **Behavior classification** via state machine
5. **Temporal smoothing** and export

All results are uploaded to GCS in the standard MouserCV schema format.

**Runtime:** Select *GPU* (T4 or better) under Runtime > Change runtime type.

## 1. Setup

In [ ]:
!pip install -q ultralytics "deeplabcut[gui,modelzoo]" google-cloud-storage \
    opencv-python-headless numpy scipy matplotlib pycocotools

## 2. Authenticate and Configure

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully.')

In [ ]:
# ============================================================
# EDITABLE PARAMETERS
# ============================================================
BUCKET = "mousercv-data"
VIDEO_ID = "test_001"
PROJECT = "demo"
NUM_MICE = 2

# SAM3 prompt boxes — one [x1, y1, x2, y2] per mouse.
PROMPT_BOXES = [
    [100, 150, 300, 350],  # Mouse 1
    [400, 200, 600, 400],  # Mouse 2
]

# DLC config
PCUTOFF = 0.3

# Feature extraction
FEATURE_WINDOW = 15  # frames for temporal features

# Behavior classification thresholds
VELOCITY_THRESHOLD = 5.0    # pixels/frame
VELOCITY_HYSTERESIS = 2.0   # hysteresis band
MIN_BOUT_FRAMES = 7         # minimum bout duration (frames)
SMOOTHING_WINDOW = 7        # median filter window

assert len(PROMPT_BOXES) == NUM_MICE

## 3. Download Video from GCS

In [ ]:
import os
import json
import cv2
import numpy as np
from google.cloud import storage

gcs_client = storage.Client()
gcs_bucket = gcs_client.bucket(BUCKET)

gcs_video_path = f"videos/{PROJECT}/{VIDEO_ID}.mp4"
local_video_path = f"/tmp/{VIDEO_ID}.mp4"

blob = gcs_bucket.blob(gcs_video_path)
blob.download_to_filename(local_video_path)

# Get video properties
cap = cv2.VideoCapture(local_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vid_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = frame_count / fps
cap.release()

print(f"Video: {vid_width}x{vid_height} @ {fps:.1f} fps, "
      f"{frame_count} frames, {duration_sec:.1f} sec")

## 4. SAM3 Segmentation and Tracking

Run SAM3 with bounding-box prompts to segment and track each mouse.

In [ ]:
from ultralytics import SAM
from pycocotools import mask as mask_utils


def encode_mask_rle(binary_mask: np.ndarray) -> str:
    """Encode a binary mask to COCO RLE format."""
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    rle["counts"] = rle["counts"].decode("utf-8")
    return json.dumps(rle)


def compute_mask_metrics(binary_mask: np.ndarray):
    """Compute area, centroid, and bounding box from a binary mask."""
    ys, xs = np.where(binary_mask > 0)
    if len(xs) == 0:
        return None
    area = int(len(xs))
    cx, cy = float(np.mean(xs)), float(np.mean(ys))
    x1, y1 = int(np.min(xs)), int(np.min(ys))
    x2, y2 = int(np.max(xs)), int(np.max(ys))
    return {
        "area": area,
        "centroid_x": cx,
        "centroid_y": cy,
        "bbox": [x1, y1, x2, y2],
    }


# Load SAM3
sam_model = SAM("sam3.pt")
print("SAM3 model loaded.")

# Run tracking
all_mask_records = []
results = sam_model.track(
    source=local_video_path,
    bboxes=PROMPT_BOXES,
    verbose=False,
    stream=True,
)

for frame_idx, result in enumerate(results):
    if result.masks is None:
        continue
    for obj_id, mask_tensor in enumerate(result.masks.data):
        binary_mask = mask_tensor.cpu().numpy().astype(np.uint8)
        metrics = compute_mask_metrics(binary_mask)
        if metrics is None:
            continue
        record = {
            "frame": frame_idx,
            "object_id": obj_id,
            "mask_rle": encode_mask_rle(binary_mask),
            "bbox": metrics["bbox"],
            "centroid_x": metrics["centroid_x"],
            "centroid_y": metrics["centroid_y"],
            "area": metrics["area"],
            "confidence": float(result.boxes.conf[obj_id])
                if result.boxes is not None else 1.0,
        }
        all_mask_records.append(record)
    if (frame_idx + 1) % 200 == 0:
        print(f"  SAM3: Processed {frame_idx + 1} frames...")

print(f"SAM3 complete: {len(all_mask_records)} mask records")

## 5. DeepLabCut SuperAnimal Keypoints

Run SuperAnimal-Quadruped zero-shot on the same video.

In [ ]:
import deeplabcut
import pandas as pd
import glob

deeplabcut.video_inference_superanimal(
    [local_video_path],
    superanimal_name="superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    video_adapt=True,
    pcutoff=PCUTOFF,
)

# Load results
h5_files = glob.glob(f"/tmp/{VIDEO_ID}*.h5")
if not h5_files:
    h5_files = glob.glob(f"/tmp/*{VIDEO_ID}*/**/*.h5", recursive=True)
if not h5_files:
    raise FileNotFoundError("No DLC h5 output found.")

dlc_df = pd.read_hdf(h5_files[0])
scorer = dlc_df.columns.get_level_values(0)[0]

if dlc_df.columns.nlevels == 4:
    bodyparts = dlc_df.columns.get_level_values(2).unique().tolist()
    individuals = dlc_df.columns.get_level_values(1).unique().tolist()
    multi_animal = True
else:
    bodyparts = dlc_df.columns.get_level_values(1).unique().tolist()
    individuals = ["animal"]
    multi_animal = False

print(f"DLC complete: {len(dlc_df)} frames, {len(bodyparts)} bodyparts, "
      f"{len(individuals)} individuals")

## 6. Feature Extraction

For each frame and each tracked object, extract geometric and temporal features
from the segmentation masks:

**Per-frame features:**
- `area`, `centroid_x`, `centroid_y`
- `aspect_ratio` (bbox height / width)
- `convexity` (area / convex hull area)
- `circularity` (4 * pi * area / perimeter^2)
- `ellipse_angle` (orientation of fitted ellipse)

**Temporal features (over sliding windows):**
- `centroid_velocity`
- `area_change_rate`
- `contour_oscillation_index` (std of convexity)
- `aspect_ratio_oscillation` (std of aspect_ratio)

In [ ]:
from scipy import ndimage
import math


def compute_per_frame_features(binary_mask: np.ndarray) -> dict | None:
    """Extract geometric features from a single binary mask."""
    ys, xs = np.where(binary_mask > 0)
    if len(xs) < 10:
        return None

    area = len(xs)
    cx, cy = float(np.mean(xs)), float(np.mean(ys))

    # Bounding box and aspect ratio
    x1, y1, x2, y2 = int(np.min(xs)), int(np.min(ys)), int(np.max(xs)), int(np.max(ys))
    bbox_w = max(x2 - x1, 1)
    bbox_h = max(y2 - y1, 1)
    aspect_ratio = bbox_h / bbox_w

    # Convex hull area
    try:
        points = np.column_stack((xs, ys))
        from scipy.spatial import ConvexHull
        hull = ConvexHull(points)
        convex_area = hull.volume  # 2D: volume = area
        convexity = area / max(convex_area, 1)
    except Exception:
        convexity = 1.0

    # Perimeter and circularity
    # Use morphological gradient for perimeter estimation
    dilated = ndimage.binary_dilation(binary_mask)
    perimeter = float(np.sum(dilated.astype(int) - binary_mask.astype(int)))
    perimeter = max(perimeter, 1)
    circularity = 4 * math.pi * area / (perimeter ** 2)

    # Ellipse angle via image moments
    try:
        contours, _ = cv2.findContours(
            binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours and len(contours[0]) >= 5:
            ellipse = cv2.fitEllipse(contours[0])
            ellipse_angle = ellipse[2]  # degrees
        else:
            ellipse_angle = 0.0
    except Exception:
        ellipse_angle = 0.0

    return {
        "area": area,
        "centroid_x": cx,
        "centroid_y": cy,
        "aspect_ratio": round(aspect_ratio, 4),
        "convexity": round(convexity, 4),
        "circularity": round(circularity, 4),
        "ellipse_angle": round(ellipse_angle, 2),
        "bbox": [x1, y1, x2, y2],
    }


def compute_temporal_features(
    features_list: list[dict], idx: int, window: int
) -> dict:
    """Compute temporal features over a sliding window."""
    start = max(0, idx - window + 1)
    window_feats = features_list[start : idx + 1]

    if len(window_feats) < 2:
        return {
            "centroid_velocity": 0.0,
            "area_change_rate": 0.0,
            "contour_oscillation_index": 0.0,
            "aspect_ratio_oscillation": 0.0,
        }

    # Centroid velocity (pixels/frame)
    cxs = [f["centroid_x"] for f in window_feats]
    cys = [f["centroid_y"] for f in window_feats]
    dx = np.diff(cxs)
    dy = np.diff(cys)
    speeds = np.sqrt(dx**2 + dy**2)
    velocity = float(np.mean(speeds))

    # Area change rate
    areas = [f["area"] for f in window_feats]
    area_changes = np.abs(np.diff(areas))
    area_change_rate = float(np.mean(area_changes))

    # Contour oscillation (std of convexity)
    convexities = [f["convexity"] for f in window_feats]
    oscillation = float(np.std(convexities))

    # Aspect ratio oscillation
    ars = [f["aspect_ratio"] for f in window_feats]
    ar_osc = float(np.std(ars))

    return {
        "centroid_velocity": round(velocity, 4),
        "area_change_rate": round(area_change_rate, 2),
        "contour_oscillation_index": round(oscillation, 6),
        "aspect_ratio_oscillation": round(ar_osc, 6),
    }


# Build per-object frame features
from collections import defaultdict

# Group mask records by object_id
obj_records = defaultdict(dict)
for rec in all_mask_records:
    obj_records[rec["object_id"]][rec["frame"]] = rec

all_feature_records = []

for obj_id in sorted(obj_records.keys()):
    frames_data = obj_records[obj_id]
    sorted_frames = sorted(frames_data.keys())

    # Decode masks and compute per-frame features
    obj_features = []
    for fidx in sorted_frames:
        rec = frames_data[fidx]
        rle = json.loads(rec["mask_rle"])
        binary = mask_utils.decode(rle)
        feats = compute_per_frame_features(binary)
        if feats is None:
            continue
        feats["frame"] = fidx
        feats["object_id"] = obj_id
        obj_features.append(feats)

    # Add temporal features
    for i, feats in enumerate(obj_features):
        temporal = compute_temporal_features(obj_features, i, FEATURE_WINDOW)
        feats.update(temporal)
        all_feature_records.append(feats)

    print(f"Object {obj_id}: {len(obj_features)} feature frames")

print(f"\nTotal feature records: {len(all_feature_records)}")

## 7. Behavior Classification (State Machine)

Three-level hierarchical classification:

- **Level 1:** MOVING vs STATIONARY (velocity threshold with hysteresis)
- **Level 2:** When STATIONARY, classify:
  - `rearing`: aspect_ratio > 1.3 AND area decrease
  - `scratching`: high oscillation + low convexity
  - `grooming`: moderate oscillation + high convexity
  - `idle`: default stationary
- **Level 3:** `uncertain` when no predicate matches confidently

In [ ]:
def classify_behavior(
    feats: dict,
    prev_state: str,
    was_moving: bool,
) -> tuple[str, bool]:
    """
    Classify a single frame's behavior using a state machine.

    Returns (behavior_label, is_moving).
    """
    velocity = feats["centroid_velocity"]
    aspect_ratio = feats["aspect_ratio"]
    convexity = feats["convexity"]
    oscillation = feats["contour_oscillation_index"]
    area_change = feats["area_change_rate"]
    ar_osc = feats["aspect_ratio_oscillation"]

    # Level 1: MOVING vs STATIONARY with hysteresis
    if was_moving:
        is_moving = velocity > (VELOCITY_THRESHOLD - VELOCITY_HYSTERESIS)
    else:
        is_moving = velocity > (VELOCITY_THRESHOLD + VELOCITY_HYSTERESIS)

    if is_moving:
        return "moving", True

    # Level 2: Stationary sub-classification
    confidence_flags = []

    # Rearing: high aspect ratio + area decrease
    if aspect_ratio > 1.3 and area_change > 50:
        confidence_flags.append(("rearing", 0.8))

    # Scratching: high oscillation + low convexity
    if oscillation > 0.05 and convexity < 0.7:
        confidence_flags.append(("scratching", 0.7))

    # Grooming: moderate oscillation + high convexity
    if 0.01 < oscillation < 0.05 and convexity > 0.75:
        confidence_flags.append(("grooming", 0.7))

    if not confidence_flags:
        # Check if anything was close
        if oscillation > 0.02 or ar_osc > 0.1:
            return "uncertain", False
        return "idle", False

    # Pick highest confidence prediction
    confidence_flags.sort(key=lambda x: x[1], reverse=True)
    best_label, best_conf = confidence_flags[0]

    # Level 3: UNCERTAIN if confidence is too low or multiple options
    if len(confidence_flags) > 1 and best_conf < 0.8:
        return "uncertain", False

    return best_label, False


# Classify all frames per object
obj_feature_map = defaultdict(list)
for feat in all_feature_records:
    obj_feature_map[feat["object_id"]].append(feat)

raw_behaviors = {}  # {obj_id: [(frame, label), ...]}

for obj_id in sorted(obj_feature_map.keys()):
    feats_list = sorted(obj_feature_map[obj_id], key=lambda x: x["frame"])
    prev_state = "idle"
    was_moving = False
    obj_behaviors = []

    for feats in feats_list:
        label, is_moving = classify_behavior(feats, prev_state, was_moving)
        obj_behaviors.append((feats["frame"], label))
        prev_state = label
        was_moving = is_moving

    raw_behaviors[obj_id] = obj_behaviors
    # Print distribution
    from collections import Counter
    dist = Counter(label for _, label in obj_behaviors)
    print(f"Object {obj_id} raw: {dict(dist)}")

print("\nRaw behavior classification complete.")

## 8. Temporal Smoothing

Apply median filter, enforce minimum bout duration, and bridge short gaps.

In [ ]:
from scipy.signal import medfilt

BEHAVIOR_LABELS = ["idle", "moving", "rearing", "scratching", "grooming", "uncertain"]
LABEL_TO_INT = {label: i for i, label in enumerate(BEHAVIOR_LABELS)}
INT_TO_LABEL = {i: label for i, label in enumerate(BEHAVIOR_LABELS)}


def smooth_behaviors(
    frame_labels: list[tuple[int, str]],
    median_window: int = 7,
    min_bout_frames: int = 7,
) -> list[tuple[int, str]]:
    """Apply temporal smoothing to behavior labels."""
    if len(frame_labels) < median_window:
        return frame_labels

    frames = [f for f, _ in frame_labels]
    labels_int = np.array([LABEL_TO_INT.get(l, 0) for _, l in frame_labels])

    # Step 1: Median filter
    smoothed_int = medfilt(labels_int, kernel_size=median_window).astype(int)

    # Step 2: Enforce minimum bout duration
    result = smoothed_int.copy()
    i = 0
    while i < len(result):
        # Find the end of the current bout
        j = i + 1
        while j < len(result) and result[j] == result[i]:
            j += 1
        bout_len = j - i
        if bout_len < min_bout_frames:
            # Replace short bout with surrounding label
            prev_label = result[i - 1] if i > 0 else result[i]
            next_label = result[j] if j < len(result) else result[i]
            replacement = prev_label if prev_label == next_label else prev_label
            result[i:j] = replacement
        i = j

    return [(frames[k], INT_TO_LABEL.get(int(result[k]), "idle")) for k in range(len(frames))]


smoothed_behaviors = {}
for obj_id, raw in raw_behaviors.items():
    smoothed = smooth_behaviors(raw, SMOOTHING_WINDOW, MIN_BOUT_FRAMES)
    smoothed_behaviors[obj_id] = smoothed
    dist = Counter(label for _, label in smoothed)
    print(f"Object {obj_id} smoothed: {dict(dist)}")

print("\nTemporal smoothing complete.")

## 9. Export Results

Export all results in the MouserCV GCS schema:
- `masks.jsonl` -- per-frame mask data
- `features.jsonl` -- per-frame feature vectors
- `behaviors.json` -- behavior segments (BehaviorExport format)
- `metadata.json` -- updated VideoMetadata

In [ ]:
from datetime import datetime, timezone

results_dir = f"/tmp/results/{VIDEO_ID}"
os.makedirs(results_dir, exist_ok=True)

# --- masks.jsonl ---
masks_path = os.path.join(results_dir, "masks.jsonl")
with open(masks_path, "w") as f:
    for rec in all_mask_records:
        f.write(json.dumps(rec) + "\n")
print(f"Wrote masks.jsonl: {len(all_mask_records)} records")

# --- features.jsonl ---
features_path = os.path.join(results_dir, "features.jsonl")
with open(features_path, "w") as f:
    for rec in all_feature_records:
        f.write(json.dumps(rec) + "\n")
print(f"Wrote features.jsonl: {len(all_feature_records)} records")

# --- behaviors.json ---
# Convert frame-level labels to segments
def labels_to_segments(
    frame_labels: list[tuple[int, str]], obj_id: int
) -> list[dict]:
    """Convert frame-level labels to contiguous behavior segments."""
    if not frame_labels:
        return []
    segments = []
    seg_start_frame, seg_label = frame_labels[0]

    for i in range(1, len(frame_labels)):
        cur_frame, cur_label = frame_labels[i]
        if cur_label != seg_label:
            prev_frame = frame_labels[i - 1][0]
            segments.append({
                "track_label": f"mouse_{obj_id}",
                "start_frame": seg_start_frame,
                "end_frame": prev_frame,
                "start_sec": round(seg_start_frame / fps, 3),
                "end_sec": round(prev_frame / fps, 3),
                "behavior": seg_label,
                "source": "state_machine",
                "confidence": 0.7,
            })
            seg_start_frame = cur_frame
            seg_label = cur_label

    # Final segment
    last_frame = frame_labels[-1][0]
    segments.append({
        "track_label": f"mouse_{obj_id}",
        "start_frame": seg_start_frame,
        "end_frame": last_frame,
        "start_sec": round(seg_start_frame / fps, 3),
        "end_sec": round(last_frame / fps, 3),
        "behavior": seg_label,
        "source": "state_machine",
        "confidence": 0.7,
    })
    return segments


all_segments = []
for obj_id, labels in smoothed_behaviors.items():
    segs = labels_to_segments(labels, obj_id)
    all_segments.extend(segs)

behaviors_path = os.path.join(results_dir, "behaviors.json")
with open(behaviors_path, "w") as f:
    json.dump(all_segments, f, indent=2)
print(f"Wrote behaviors.json: {len(all_segments)} segments")

# --- metadata.json ---
metadata = {
    "video_id": VIDEO_ID,
    "filename": f"{VIDEO_ID}.mp4",
    "project_name": PROJECT,
    "dataset_name": None,
    "subject_count": NUM_MICE,
    "fps": fps,
    "duration_sec": round(duration_sec, 2),
    "width": vid_width,
    "height": vid_height,
    "camera_angle": "front_angled",
    "calibration_px_per_cm": None,
    "notes": "Processed via MouserCV Colab pipeline",
    "tags": ["colab", "sam3", "superanimal"],
    "gcs_video_uri": f"gs://{BUCKET}/videos/{PROJECT}/{VIDEO_ID}.mp4",
    "gcs_results_uri": f"gs://{BUCKET}/results/{VIDEO_ID}/",
    "processing_status": "ready",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "processed_at": datetime.now(timezone.utc).isoformat(),
}

metadata_path = os.path.join(results_dir, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Wrote metadata.json")

## 10. Upload All Results to GCS

In [ ]:
# Upload all result files
result_files = ["masks.jsonl", "features.jsonl", "behaviors.json", "metadata.json"]

for filename in result_files:
    local_path = os.path.join(results_dir, filename)
    if filename == "metadata.json":
        gcs_path = f"metadata/{VIDEO_ID}.json"
    else:
        gcs_path = f"results/{VIDEO_ID}/{filename}"
    blob = gcs_bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    size_kb = os.path.getsize(local_path) / 1024
    print(f"  Uploaded {filename} -> gs://{BUCKET}/{gcs_path} ({size_kb:.1f} KB)")

print("\nAll results uploaded to GCS.")

## 11. Processing Summary

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("PROCESSING SUMMARY")
print("=" * 60)
print(f"Video:              {VIDEO_ID}")
print(f"Duration:           {duration_sec:.1f} sec ({frame_count} frames @ {fps:.1f} fps)")
print(f"Resolution:         {vid_width}x{vid_height}")
print(f"Mice tracked:       {NUM_MICE}")
print(f"Mask records:       {len(all_mask_records)}")
print(f"Feature records:    {len(all_feature_records)}")
print(f"Behavior segments:  {len(all_segments)}")
print()

# Behavior distribution
behavior_dist = Counter(seg["behavior"] for seg in all_segments)
print("Behavior distribution (segments):")
for beh, count in behavior_dist.most_common():
    # Calculate total time for this behavior
    total_time = sum(
        seg["end_sec"] - seg["start_sec"]
        for seg in all_segments if seg["behavior"] == beh
    )
    print(f"  {beh:15s}  {count:4d} segments  {total_time:7.1f} sec")

# Pie chart
fig, ax = plt.subplots(figsize=(8, 8))
labels = list(behavior_dist.keys())
sizes = list(behavior_dist.values())
colors_pie = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#95a5a6"]
ax.pie(sizes, labels=labels, colors=colors_pie[:len(labels)],
       autopct="%1.1f%%", startangle=90)
ax.set_title(f"Behavior Distribution - {VIDEO_ID}")
plt.tight_layout()
plt.show()